# 07: GhanaTok Vocabulary Size Sweep (4,000, 8,000, 16,000, 32,000)

**Objective**: Systematically train GhanaTok (both Byte-level BPE and SentencePiece Unigram) across four vocabulary sizes (4k, 8k, 16k, 32k), evaluate each on the held-out test sets for Twi and Ewe, and empirically validate the 8,000-token design choice.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import PreTrainedTokenizerFast

sys.path.append("..")
from src.train_ghanatok import (
    train_ghanatok,
    train_ghanatok_unigram,
    SentencePieceWrapper,
    PROJECT_ROOT,
)
from src.tokenizer_benchmark import run_benchmark

VOCAB_SIZES = [4000, 8000, 16000, 32000]

## 2. Load Sweep Tokenizers
We load all 8 trained models from `models/ghanatok_{bpe,unigram}_{vocab_size}/`.

In [ ]:
sweep_tokenizers = {}
for vocab_size in VOCAB_SIZES:
    bpe_path = PROJECT_ROOT / "models" / f"ghanatok_bpe_{vocab_size}" / "ghanatok_tokenizer.json"
    sweep_tokenizers[f"BPE-{vocab_size}"] = PreTrainedTokenizerFast(
        tokenizer_file=str(bpe_path),
        unk_token="<unk>", pad_token="<pad>", bos_token="<s>", eos_token="</s>",
    )

    unigram_path = PROJECT_ROOT / "models" / f"ghanatok_unigram_{vocab_size}" / "ghanatok_unigram.model"
    sweep_tokenizers[f"Unigram-{vocab_size}"] = SentencePieceWrapper(str(unigram_path))

print("Loaded tokenizers:", list(sweep_tokenizers.keys()))

## 3. Load Summary Results & Display Table
We load the aggregated sweep results generated from the held-out test sets (`data/processed/twi/test.txt` and `data/processed/ewe/test.txt`).

In [ ]:
summary_csv = PROJECT_ROOT / "reports" / "tables" / "vocab_size_sweep_summary.csv"
sweep_summary = pd.read_csv(summary_csv)
display(sweep_summary)

## 4. Step-wise Relative Improvements
We calculate the relative percentage fertility reduction between consecutive vocabulary sizes (4k -> 8k, 8k -> 16k, 16k -> 32k).

In [ ]:
for language in ["twi", "ewe"]:
    for algorithm in ["BPE", "Unigram"]:
        subset = sweep_summary[
            (sweep_summary["language"] == language) & (sweep_summary["algorithm"] == algorithm)
        ].sort_values("vocab_size")
        fert = subset.set_index("vocab_size")["mean_fertility"]
        print(f"\n{language.upper()} {algorithm}:")
        sizes = list(fert.index)
        for i in range(1, len(sizes)):
            prev_size, curr_size = sizes[i-1], sizes[i]
            prev_fert, curr_fert = fert[prev_size], fert[curr_size]
            pct_change = (prev_fert - curr_fert) / prev_fert * 100
            print(f"  {prev_size:>5} -> {curr_size:>5}: fertility {prev_fert:.3f} -> {curr_fert:.3f} ({pct_change:+.2f}%)")

## 5. Visualizing Fertility vs Vocabulary Size

In [ ]:
from IPython.display import Image
figure_path = PROJECT_ROOT / "reports" / "figures" / "vocab_size_sweep_fertility.png"
Image(filename=str(figure_path))